# Enemy & Behavior : Game NPC Pathfinding Using A*


- **PATROL:** enemy bakal gerak random selama player belum kedeteksi.
- **CHASE:** kalau player udah kedeteksi, enemy bakal ngikutin jalur dari A*, tapi cuma maju satu langkah tiap giliran.
- **CAUGHT:** kalau posisi enemy sama kayak player, berarti player ketangkep.
- **Turn-based:** player jalan duluan, baru setelah itu enemy gerak sesuai kondisinya.
- **8 arah:** enemy bisa gerak ke atas, bawah, kiri, kanan, sama diagonal, sesuai konsep kita di slide 11.





**Aturan jalur:** `path[0]` itu posisi enemy sekarang, terus koordinat setelahnya adalah langkah yang bakal dilewati. Setiap giliran, enemy cuma bergerak ke `path[1]`. Kalau player pindah posisi, program utama perlu manggil A* lagi buat cari jalur baru.

**Catatan:** buat konversi dan aturan deteksinya nanti disesuaiin sama yang bagian LOS tapi disini aku pake dummy grid sama simulasi manual

**Gerak diagonal:** disini enemy nggak bisa jalan diagonal nembus sudut rintangan. ada di `is_valid_step()`.

## 1. Helper pergerakan dan validasi rintangan

In [ ]:

import random

# Enemy bisa gerak ke 8 arah: atas, bawah, kiri, kanan, + diagonal.
DIRECTIONS_8 = [
    (-1, 0), (1, 0), (0, -1), (0, 1),
    (-1, -1), (-1, 1), (1, -1), (1, 1)
]

# TODO KHEN: ini simbol tembok sementara, nanti samain sama grid.
WALL = "#"


def is_walkable(grid, position):
    """Cek apakah posisi masih di grid dan bukan tembok."""
    row, col = position

    return (
        0 <= row < len(grid)
        and 0 <= col < len(grid[row])
        and grid[row][col] != WALL
    )


def is_valid_step(grid, old_pos, new_pos):
    """Cek apakah enemy boleh pindah ke posisi baru."""
    
    # Kalau posisi awal atau tujuan nggak valid gak bisa jalan
    if not is_walkable(grid, old_pos) or not is_walkable(grid, new_pos):
        return False

    # Hitung selisih posisi lama sama posisi baru.
    dr = new_pos[0] - old_pos[0]
    dc = new_pos[1] - old_pos[1]

    # Sekali jalan cuma boleh geser 1 kotak, termasuk diagonal.
    if max(abs(dr), abs(dc)) != 1:
        return False

    # Kalau geraknya diagonal, jangan sampai motong sudut tembok.
    # Jadi dua kotak di sampingnya juga harus kosong.
    if dr != 0 and dc != 0:
        side_1 = (old_pos[0] + dr, old_pos[1])
        side_2 = (old_pos[0], old_pos[1] + dc)

        if not (is_walkable(grid, side_1) and is_walkable(grid, side_2)):
            return False

    return True


def get_valid_neighbors(grid, position):
    """Cari semua kotak di sekitar enemy yang bisa dilewati."""
    row, col = position

    # Cek semua kemungkinan gerak dari 8 arah.
    candidates = [(row + dr, col + dc) for dr, dc in DIRECTIONS_8]

    # Ambil yang valid aja, jangan sampai nabrak tembok.
    return [pos for pos in candidates if is_valid_step(grid, position, pos)]



## 2. Class `Enemy`

In [3]:

class Enemy:
    """Class buat ngatur enemy, mulai dari PATROL, CHASE, sampai CAUGHT."""

    def __init__(self, enemy_id, position, seed=None):
        self.enemy_id = enemy_id
        self.position = tuple(position)  # Posisi enemy (row, col)
        self.state = "PATROL"
        self.rng = random.Random(seed)    # Biar gerak random pas demo bisa diulang

    def patrol(self, grid):
        """Kalau player belum kedeteksi, enemy bakal jalan random."""
        options = get_valid_neighbors(grid, self.position)

        # Kalau nggak ada jalan yang bisa dilewati, enemy diem aja.
        if not options:
            return "PATROL: nggak ada jalan yang bisa dilewati, enemy diem."

        # Pilih salah satu kotak yang aman secara random.
        self.position = self.rng.choice(options)
        return f"PATROL: enemy jalan random ke {self.position}."

    def chase(self, grid, path):
        """Enemy ngikutin satu langkah dari jalur A* yang udah dihitung temen."""

        # TODO KHEN:
            # path A* nya
            

        # Kalau belum ada jalur atau jalurnya terlalu pendek, enemy diem.
        if path is None or len(path) < 2:
            return "CHASE: belum ada jalur dari A*, jadi enemy diem."

        path = [tuple(cell) for cell in path]

        # Pastikan jalurnya dimulai dari posisi enemy sekarang.
        if path[0] != self.position:
            return "CHASE: posisi awal path beda sama posisi enemy, jadi diem dulu."

        # Enemy cuma ambil satu langkah berikutnya, bukan langsung ke player.
        next_pos = path[1]

        # Cek dulu apakah langkah berikutnya aman buat dilewati.
        if not is_valid_step(grid, self.position, next_pos):
            return f"CHASE: jalan ke {next_pos} nggak valid atau ketutup tembok."

        self.position = next_pos
        return f"CHASE: enemy ngikutin A* ke {self.position}."

    def check_caught(self, player_position):
        """Kalau posisi player sama kayak enemy, berarti ketangkep."""
        return self.position == tuple(player_position)

    def update(self, grid, player_position, player_detected, path=None):
        """Dipanggil tiap giliran setelah player selesai gerak.

        Data yang nanti dikirim dari program utama/Khen:
        - grid            : map dari bagian grid
        - player_position : posisi player sekarang
        - player_detected : True kalau player kedeteksi, False kalau nggak
        - path            : jalur dari A*, kalau nggak ada ya None
        """

        # Simpan posisi dan state sebelumnya buat dibandingin pas print.
        before_pos = self.position
        before_state = self.state

        if self.state == "CAUGHT":
            # Kalau udah ketangkep, enemy nggak perlu ngapa-ngapain lagi.
            detail = "Player udah ketangkep, enemy nggak gerak lagi."

        elif self.check_caught(player_position):
            # Bisa aja player malah jalan ke kotak tempat enemy berdiri.
            self.state = "CAUGHT"
            detail = "Player masuk ke posisi enemy."

        elif player_detected:
            # Kalau player kelihatan, enemy mulai ngejar pakai path A*.
            self.state = "CHASE"
            detail = self.chase(grid, path)

        else:
            # Kalau player nggak kelihatan, enemy balik jalan random.
            self.state = "PATROL"
            detail = self.patrol(grid)

        # Cek lagi setelah enemy gerak, siapa tahu udah nyampe ke player.
        if self.check_caught(player_position):
            self.state = "CAUGHT"
            detail += " PLAYER KETANGKEP!"

        # Print biar gampang lihat enemy lagi ngapain tiap giliran.
        print(f"[{self.enemy_id}] {before_state} -> {self.state}")
        print(f"  Posisi: {before_pos} -> {self.position}")
        print(f"  Player: {tuple(player_position)} | Kedeteksi: {player_detected}")
        print(f"  Aksi  : {detail}")

        # Balikin hasilnya biar bisa dipakai program utama.
        return {
            "enemy_id": self.enemy_id,
            "position": self.position,
            "state": self.state,
            "caught": self.state == "CAUGHT",
            "description": detail,
        }



## 3. Dummy grid buat nguji

Grid yang di bawah ini cuma buat nyoba kode di notebook, jadi belum grid final yang bakal kita pakai. Buat grid sama rintangannya nanti tetep ngikutin punya temen yang ngerjain bagian itu.

`render_grid()` juga cuma buat nampilin map lewat `print()`, bukan buat bikin grid dari awal.

In [4]:

# Grid sementara buat nyobain enemy, ntar diganti pake grid dr temen
demo_grid = [
    ".......",
    "..#....",
    "..#....",
    ".......",
    ".......",
]


def render_grid(grid, player_position, enemies):
    """Buat nampilin map pake print aja, blm GUI final"""

    # Ubah grid jd list biar bisa nampilin posisi player sm enemy
    canvas = [list(row) for row in grid]

    # Masukin posisi tiap enemy ke map
    for enemy in enemies:
        r, c = enemy.position

        # Cek dulu posisinya valid atau engga
        if is_walkable(grid, (r, c)):
            canvas[r][c] = "E"

    # Ambil posisi player
    pr, pc = player_position

    if is_walkable(grid, (pr, pc)):
        # Kalo player sm enemy ada di kotak yg sama, tandain X
        # Kalo engga, ya tetep P
        canvas[pr][pc] = "X" if canvas[pr][pc] == "E" else "P"

    # Print map per baris
    for row in canvas:
        print(" ".join(row))

    print("P = player | E = enemy | # = tembok | X = ketangkep")


# Coba nampilin grid pake 1 enemy dulu
render_grid(demo_grid, player_position=(3, 4), enemies=[Enemy("E1", (3, 1))])


. . . . . . .
. . # . . . .
. . # . . . .
. E . . P . .
. . . . . . .
P = player | E = enemy | # = tembok | X = ketangkep


## 4. Demo A : PATROL pas player blm keliatan

`player_detected=False` di bawah ini cuma simulasi hasil deteksi aja, bukan dr fungsi line-of-sight yg kita bikin sendiri

In [5]:
patrol_enemy = Enemy("E-PATROL", (0, 0), seed=5)

for turn in range(1, 4):
    print(f"\n=== GILIRAN PATROL {turn} ===")
    patrol_enemy.update(
        grid=demo_grid,
        player_position=(0, 6),
        player_detected=False,
        path=None,
    )
    render_grid(demo_grid, (0, 6), [patrol_enemy])


=== GILIRAN PATROL 1 ===
[E-PATROL] PATROL -> PATROL
  Posisi: (0, 0) -> (1, 1)
  Player: (0, 6) | Kedeteksi: False
  Aksi  : PATROL: enemy jalan random ke (1, 1).
. . . . . . P
. E # . . . .
. . # . . . .
. . . . . . .
. . . . . . .
P = player | E = enemy | # = tembok | X = ketangkep

=== GILIRAN PATROL 2 ===
[E-PATROL] PATROL -> PATROL
  Posisi: (1, 1) -> (1, 0)
  Player: (0, 6) | Kedeteksi: False
  Aksi  : PATROL: enemy jalan random ke (1, 0).
. . . . . . P
E . # . . . .
. . # . . . .
. . . . . . .
. . . . . . .
P = player | E = enemy | # = tembok | X = ketangkep

=== GILIRAN PATROL 3 ===
[E-PATROL] PATROL -> PATROL
  Posisi: (1, 0) -> (1, 1)
  Player: (0, 6) | Kedeteksi: False
  Aksi  : PATROL: enemy jalan random ke (1, 1).
. . . . . . P
. E # . . . .
. . # . . . .
. . . . . . .
. . . . . . .
P = player | E = enemy | # = tembok | X = ketangkep


## 5. Demo B — mode CHASE, target bergerak, terus CAUGHT

Koordinat di `path` ini cuma contoh hasil A* yg dibikin manual buat ngetes enemy, jadi ga perlu nunggu A* dr temen dulu

Enemy juga ga nyari jalurnya sendiri, cuma ngikutin langkah berikutnya dr `path[1]`

In [6]:

# Bikin enemy buat ngetes CHASE
chase_enemy = Enemy("E-CHASE", (3, 1), seed=7)

# Simulasi 3 turn, player pindah jd path A* ikut berubah
# Path masih manual, ntar diganti hasil A* dr Khen
scenarios = [
    {
        # Turn 1: enemy mulai ngejar player dr (3,1)
        "player": (3, 4),
        "detected": True,
        "path": [(3, 1), (3, 2), (3, 3), (3, 4)],
    },
    {
        # Turn 2: player pindah, path diperbarui
        "player": (2, 4),
        "detected": True,
        "path": [(3, 2), (3, 3), (2, 4)],
    },
    {
        # Turn 3: enemy nyampe ke player, jd CAUGHT
        "player": (3, 4),
        "detected": True,
        "path": [(3, 3), (3, 4)],
    },
]

# Jalanin simulasi per turn
for turn, scenario in enumerate(scenarios, start=1):
    print(f"\n=== TURN CHASE {turn} ===")
    print("Contoh path dr A*:", scenario["path"])

    # Update enemy, tiap turn cuma maju 1 langkah
    result = chase_enemy.update(
        grid=demo_grid,
        player_position=scenario["player"],
        player_detected=scenario["detected"],
        path=scenario["path"],
    )

    # Nampilin posisi terbaru
    render_grid(demo_grid, scenario["player"], [chase_enemy])

    # Stop kalo player udh ketangkep
    if result["caught"]:
        print("GAME OVER — PLAYER KETANGKEP")
        break



=== TURN CHASE 1 ===
Contoh path dr A*: [(3, 1), (3, 2), (3, 3), (3, 4)]
[E-CHASE] PATROL -> CHASE
  Posisi: (3, 1) -> (3, 2)
  Player: (3, 4) | Kedeteksi: True
  Aksi  : CHASE: enemy ngikutin A* ke (3, 2).
. . . . . . .
. . # . . . .
. . # . . . .
. . E . P . .
. . . . . . .
P = player | E = enemy | # = tembok | X = ketangkep

=== TURN CHASE 2 ===
Contoh path dr A*: [(3, 2), (3, 3), (2, 4)]
[E-CHASE] CHASE -> CHASE
  Posisi: (3, 2) -> (3, 3)
  Player: (2, 4) | Kedeteksi: True
  Aksi  : CHASE: enemy ngikutin A* ke (3, 3).
. . . . . . .
. . # . . . .
. . # . P . .
. . . E . . .
. . . . . . .
P = player | E = enemy | # = tembok | X = ketangkep

=== TURN CHASE 3 ===
Contoh path dr A*: [(3, 3), (3, 4)]
[E-CHASE] CHASE -> CAUGHT
  Posisi: (3, 3) -> (3, 4)
  Player: (3, 4) | Kedeteksi: True
  Aksi  : CHASE: enemy ngikutin A* ke (3, 4). PLAYER KETANGKEP!
. . . . . . .
. . # . . . .
. . # . . . .
. . . . X . .
. . . . . . .
P = player | E = enemy | # = tembok | X = ketangkep
GAME OVER — PLAYE

## 6. Demo C — mendukung beberapa enemy sekaligus

Sesuai ide di slide 11, jumlah enemy bisa ditentuin di awal game. Jadi tinggal masukin beberapa `Enemy` ke satu list, terus tiap enemy di-update satu per satu

Buat deteksi sama path A* nanti nyambung ke bagian Khen

In [7]:

# Bikin 2 enemy buat contoh
enemies = [
    Enemy("E1", (4, 0), seed=1),
    Enemy("E2", (4, 6), seed=2),
]

player_position = (0, 6)

print("=== UPDATE 2 ENEMY DALAM 1 TURN ===")

# Update tiap enemy satu per satu
for enemy in enemies:
    # TODO KHEN: detected ntar dr fungsi deteksi asli
    detected = False

    # TODO KHEN: kalo kedeteksi, path diambil dr A*
    path = None

    enemy.update(demo_grid, player_position, detected, path)

# Nampilin posisi semua enemy setelah gerak
render_grid(demo_grid, player_position, enemies)


=== UPDATE 2 ENEMY DALAM 1 TURN ===
[E1] PATROL -> PATROL
  Posisi: (4, 0) -> (3, 0)
  Player: (0, 6) | Kedeteksi: False
  Aksi  : PATROL: enemy jalan random ke (3, 0).
[E2] PATROL -> PATROL
  Posisi: (4, 6) -> (3, 6)
  Player: (0, 6) | Kedeteksi: False
  Aksi  : PATROL: enemy jalan random ke (3, 6).
. . . . . . P
. . # . . . .
. . # . . . .
E . . . . . E
. . . . . . .
P = player | E = enemy | # = tembok | X = ketangkep


## 7. TES

In [8]:

# Tes dulu geraknya valid atau gak
assert is_walkable(demo_grid, (3, 2))                # bisa dilewatin
assert not is_walkable(demo_grid, (1, 2))            # ada tembok
assert is_valid_step(demo_grid, (3, 1), (3, 2))      # geser kanan, aman
assert not is_valid_step(demo_grid, (1, 1), (1, 2))  # nabrak tembok
assert not is_valid_step(demo_grid, (0, 1), (1, 2))  # diagonal tp ada tembok
assert not is_valid_step(demo_grid, (3, 1), (3, 4))  # ga bisa lompat 3 kotak

# Tes enemy pas ngejar player
unit_enemy = Enemy("E-TEST", (3, 3))

# Player ada di sebelah enemy, jadi tinggal maju 1 langkah
check = unit_enemy.update(
    demo_grid, (3, 4), True, [(3, 3), (3, 4)]
)

# Pastiin player beneran ketangkep
assert check["caught"] is True
assert check["state"] == "CAUGHT"
assert unit_enemy.position == (3, 4)

print("\nYES.")


[E-TEST] PATROL -> CAUGHT
  Posisi: (3, 3) -> (3, 4)
  Player: (3, 4) | Kedeteksi: True
  Aksi  : CHASE: enemy ngikutin A* ke (3, 4). PLAYER KETANGKEP!

YES.


## ALUR

```text
Setelah player bergerak (1 turn)
           |
           v
    Enemy.update(...)
           |
  Sudah di posisi player? -- YA --> CAUGHT / GAME OVER
           |
         TIDAK
           |
   Player terdeteksi?
       /       \
    TIDAK       YA
      |          |
   PATROL      CHASE
  gerak acak    ambil path A* dari modul teman
  valid grid    jalan SATU langkah
       \        /
        cek posisi sama?
           |
         CAUGHT jika sama
```

Buat behavior sama nentuin state enemy udh diatur di class `Enemy`, sedangkan nyari jalurnya tetep pake A* dan deteksi player dr modul line-of-sight
